# uv: Fast Python Package Management

## The Core Question

You're setting up a new CFD project. `pip install` takes 3 minutes. Your teammate's environment is broken. The lock file is a mess. How do you get a **fast, reproducible** Python environment?

**uv** is a Python package installer and resolver written in Rust. It's **10-100x faster than pip** and handles virtual environments, locking, and project management in one tool.

## Why uv?

- **Blazing fast**: written in Rust, resolves dependencies in milliseconds (not minutes)
- **All-in-one**: replaces `pip`, `pip-tools`, `virtualenv`, `pyenv`, and `pipx`
- **Reproducible**: `uv.lock` pins exact versions (like `poetry.lock`)
- **Python version management**: `uv python install 3.12` — no pyenv needed

## The Workflow

```
uv init my_cfd_project
        |
        v
  uv add numpy scipy matplotlib
        |
        v
  uv lock  (creates uv.lock with exact versions)
        |
        v
  uv sync  (creates .venv with locked versions)
        |
        v
  uv run python main.py  (runs in the venv)
```

## Setup


In [ ]:
import subprocess
import shutil

def check_uv():
    try:
        result = subprocess.run(['uv', '--version'], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            return result.stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return None

uv_version = check_uv()
if uv_version:
    print(f'uv is installed: {uv_version}')
    HAS_UV = True
else:
    print('uv is NOT installed.')
    print('Install: https://docs.astral.sh/uv/getting-started/installation/')
    print('  Windows: powershell -c "irm https://astral.sh/uv/install.ps1 | iex"')
    print('  macOS:   curl -LsSf https://astral.sh/uv/install.sh | sh')
    print()
    print('This notebook will show the commands and expected output.')
    HAS_UV = False

## 1. Initialize a Project

**Concept**: `uv init` creates a minimal project structure: `pyproject.toml`, `.python-version`, and a `main.py`. No boilerplate, no `setup.py`.


In [ ]:
if HAS_UV:
    import os
    test_dir = './uv_demo'
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    
    result = subprocess.run(['uv', 'init', test_dir], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    
    for f in sorted(os.listdir(test_dir)):
        print(f'  {f}')
else:
    print('Expected output of `uv init my_cfd_project`:')
    print()
    print('  my_cfd_project/')
    print('    pyproject.toml')
    print('    .python-version')
    print('    main.py')
    print('    README.md')

## 2. Add Dependencies

**Concept**: `uv add` adds a dependency to `pyproject.toml` AND resolves it into `uv.lock` in one step. No separate `pip install` needed.


In [ ]:
if HAS_UV:
    for dep in ['numpy', 'scipy', 'matplotlib']:
        result = subprocess.run(['uv', 'add', dep], capture_output=True, text=True, cwd='./uv_demo')
        status = 'OK' if result.returncode == 0 else 'FAIL'
        print(f'  uv add {dep:12s} -> {status}')
    
    with open('./uv_demo/pyproject.toml', 'r') as f:
        print()
        print('=== pyproject.toml ===')
        print(f.read())
    
    with open('./uv_demo/uv.lock', 'r') as f:
        lock_lines = f.readlines()
    print('=== uv.lock (first 20 lines) ===')
    print(''.join(lock_lines[:20]))
    print(f'... ({len(lock_lines)} total lines)')
else:
    print('Expected output of `uv add numpy scipy matplotlib`:')
    print()
    print('  Resolved 3 packages in 45ms')
    print('  Added numpy==2.2.1 to dependencies')
    print('  Added scipy==1.15.0 to dependencies')
    print('  Added matplotlib==3.10.0 to dependencies')
    print()
    print('pyproject.toml now has:')
    print('  dependencies = ["numpy>=2.2.1", "scipy>=1.15.0", "matplotlib>=3.10.0"]')
    print()
    print('uv.lock pins exact versions + hashes for reproducibility.')

## 3. Sync and Run

**Concept**: `uv sync` creates a `.venv` with the exact locked versions. `uv run` executes a command inside that venv. No `source .venv/bin/activate` needed.


In [ ]:
if HAS_UV:
    result = subprocess.run(['uv', 'sync'], capture_output=True, text=True, cwd='./uv_demo')
    print(result.stdout)
    
    result = subprocess.run(
        ['uv', 'run', 'python', '-c', 'import numpy; print(f"numpy {numpy.__version__} works")'],
        capture_output=True, text=True, cwd='./uv_demo'
    )
    print(result.stdout)
    
    shutil.rmtree('./uv_demo')
    print('Cleaned up demo project.')
else:
    print('Expected output of `uv sync`:')
    print()
    print('  Using CPython 3.12.4')
    print('  Creating virtual environment at .venv')
    print('  Installed numpy-2.2.1')
    print('  Installed scipy-1.15.0')
    print('  Installed matplotlib-3.10.0')
    print('  3 packages installed in 1.2s')
    print()
    print('Expected output of `uv run python -c "import numpy; print(numpy.__version__)"`:')
    print()
    print('  2.2.1')
    print()
    print('Note: no `source .venv/bin/activate` needed.')
    print('uv run automatically uses the project venv.')

## 4. Why This Matters for Engineering

**Concept**: The #1 problem in scientific computing is **"it works on my machine"**. uv solves this with:

1. **`uv.lock`**: exact versions + hashes. Your teammate runs `uv sync` and gets the SAME environment.
2. **`.python-version`**: pins the Python version. No more "I used 3.10, you used 3.12" bugs.
3. **Speed**: setting up a new environment takes seconds, not minutes. You'll actually do it.


In [ ]:
comparison = (
    "                    pip          uv\n"
    "                  --------     ---------\n"
    "  Resolve deps    30-120s      50-200ms\n"
    "  Install         60-300s      2-10s\n"
    "  Create venv     5-10s        1-2s\n"
    "  Total           2-7 min      3-12s\n"
    "\n"
    "  Lock file       pip-tools    uv.lock (built-in)\n"
    "  Python mgmt     pyenv        uv python (built-in)\n"
    "  Run in venv     source ...   uv run (built-in)\n"
)
print(comparison)
print('For a project you set up once a year, pip is fine.')
print('For a project you iterate on daily, uv saves hours.')

## Exercises

1. **Set up a real project**: Create a `uv` project for your CFD post-processing code. Add `numpy`, `scipy`, `matplotlib`, `pandas`. Run `uv sync` and time it.

2. **Reproducibility test**: Commit `pyproject.toml` + `uv.lock` to git. On a different machine, run `uv sync`. Do you get the exact same versions?

3. **Python version**: Use `uv python install 3.11` and set `.python-version` to 3.11. Does your code still work? What breaks?

4. **vs poetry**: Create the same project with both `uv` and `poetry`. Compare: setup time, lock file size, and ease of adding a dependency. Which do you prefer?

5. **CI integration**: Add a GitHub Actions step that uses `uv` instead of `pip`. How much faster is the CI?
